In [2]:
import pandas as pd
import numpy as np
import warnings
import unicodedata
from pathlib import Path
import re
from datetime import datetime
import random

# Configurar la opción para evitar la advertencia
pd.set_option('future.no_silent_downcasting', True)
2
# Leer el archivo original con TODAS las columnas como texto
archivo = Path('/Users/jhonvilcaranatintaya/egresados_empr.xlsx')
nombre_salida = archivo.with_stem(archivo.stem + "_final")
df = pd.read_excel(archivo, dtype=str)  # ← esta línea es clave

# Reemplazar "-" por NaN y evitar FutureWarning
with warnings.catch_warnings():
    warnings.simplefilter(action='ignore', category=FutureWarning)
    df = df.replace("-", np.nan).infer_objects(copy=False)
    
# Mapeo explícito de columnas a renombrar
mapeo_columnas = {
    'TIPO_EGRESADO': 'TIPO_EGRESADO',
    'Tipo de Documento': 'TIPO_DOCUMENTO',
    'N° Documento': 'NRO_DOCUMENTO',
    'Facultad': 'CODIGO_FACULTAD_UNIDAD',
    'Escuela': 'CODIGO_PROGRAMA',
    'PERIODO_LECTIVO': 'PERIODO_LECTIVO',
    'TIPO_PERIODO': 'TIPO_PERIODO',
    'Periodo de Egreso': 'PERIODO_EGRESO',
    'Fecha de Egreso': 'FECHA_EGRESO',
    'Total de créditos aprobados': 'TOTAL_CREDITOS_APROBADOS',
    'Nota promedio': 'NOTA_PROMEDIO',
    'Celular': 'CELULAR',
    'Correo personal': 'CORREO_PERSONAL',
    'Correo institucional': 'CORREO_INSTITUCIONAL',
    'Idioma extranjero acreditado': 'IDIOMA_EXTRANJERO_ACREDITADO',
    'Idioma Nativo Acreditado': 'CREDITOS_MATRICULADOS_DEL_PERIODO_ANTERIOR',
    'CODIGO_LOCAL': 'CODIGO_LOCAL',
    'Modalidad de Estudio': 'MODALIDAD_ESTUDIO',
    'Periodo de Primera Matrícula': 'PERIODO_PRIMERA_MATRICULA',
    'Fecha de Primera Matrícula': 'FECHA_PRIMERA_MATRICULA',

}

# Lista final de columnas en el orden deseado
columnas_ordenadas = [
    'TIPO_EGRESADO',
    'TIPO_DOCUMENTO',
    'NRO_DOCUMENTO',        
    'CODIGO_FACULTAD_UNIDAD',
    'CODIGO_PROGRAMA',
    'PERIODO_LECTIVO',
    'TIPO_PERIODO',
    'PERIODO_EGRESO',
    'FECHA_EGRESO',
    'TOTAL_CREDITOS_APROBADOS',
    'NOTA_PROMEDIO',
    'CELULAR',
    'CORREO_PERSONAL',
    'CORREO_INSTITUCIONAL',
    'IDIOMA_EXTRANJERO_ACREDITADO',
    'IDIOMA_NATIVO_ACREDITADO',
    'CODIGO_LOCAL',
    'MODALIDAD_ESTUDIO',
    'PERIODO_PRIMERA_MATRICULA',
    'FECHA_PRIMERA_MATRICULA'
]


# Renombrar columnas que están en el mapeo
df.rename(columns=mapeo_columnas, inplace=True)

# Añadir columnas faltantes como vacías
for col in columnas_ordenadas:
    if col not in df.columns:
        df[col] = None

# Reordenar columnas
df = df[columnas_ordenadas]

# Asignar valores fijos
df['TIPO_EGRESADO'] = 1
df['CODIGO_FACULTAD_UNIDAD'] = 9
df['CODIGO_LOCAL'] = 'SL01' 
df['PERIODO_LECTIVO'] = 1
df['TIPO_PERIODO'] = 1 

tipo_documento_map = {
    "DNI": 1,
    "PASAPORTE": 2,
    "CE": 3,
    "CÉDULA DE IDENTIDAD": 4,
    "DIDEXT": 5,
    "DIDExt": 5,
    "PERMISO TEMPORAL DE PERMANENCIA": 6,
    "CARNÉ DE IDENTIDAD": 7,
    "CÉDULA DE CIUDADANÍA": 8,
    "CARNÉ TEMPORAL DE PERMANENCIA": 9
}
# Aplicar el reemplazo en TIPO_DOCUMENTO
df['TIPO_DOCUMENTO'] = df['TIPO_DOCUMENTO'].replace(tipo_documento_map)


# Limpiar los datos (eliminar espacios y normalizar caracteres)
df['CODIGO_PROGRAMA'] = df['CODIGO_PROGRAMA'].apply(lambda x: unicodedata.normalize('NFKD', str(x)).encode('ASCII', 'ignore').decode('ASCII'))

# Asegurarse de que la columna 'CODIGO_PROGRAMA_INGRESO' es de tipo str
df['CODIGO_PROGRAMA'] = df['CODIGO_PROGRAMA'].astype(str)

# Diccionario de reemplazos
reemplazos = {
    #SEGUNDAS ESPECIALIDADES
    "Posgrado de Segunda Especialidad de Enfermeria en Emergencias y Desastres": 67,
    "Posgrado de Segunda Especialidad de Enfermeria en Gineco Obstetricia": 244,
    "Posgrado de Segunda Especialidad en Enfermeria en Administracion y Gestion": 63,
    "Posgrado de Segunda Especialidad en Enfermeria en Centro Quirurgico": 73,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos Neonatales": 75,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos Pediatricos": 76,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos": 77,
    "Posgrado de Segunda Especialidad en Enfermeria en Neurologia y Neurocirugia": 82,
    "Posgrado de Segunda Especialidad en Estadistica Aplicada para Investigacion": 87,
    "Posgrado de Segunda Especialidad de Enfermeria en Oncologia": 83,
    "Posgrado de Segunda Especialidad en Estadistica Aplicada para Investigacion": 87,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidado Integral Infantil": 74,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos": 66,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Recuperacion Posanastesica": 220,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Traumatologia y Ortopedia": 79,
    "Posgrado de Segunda Especialidad en Enfermeria en Gestion de Central de Esterilizacion": 80,
    "Posgrado de Segunda Especialidad en Salud Ocupacional": 91,
    "Posgrado de Segunda Especialidad en Enfermeria en Pediatria": 84,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Traumatologia y Ortopedia": 79,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Quirurgicos con mencion en Recuperacion Posanastesica": 220,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidados Intensivos": 66,
    "Posgrado de Segunda Especialidad en Enfermeria en Cuidado Integral Infantil": 74,
    "Posgrado de Segunda Especialidad en Salud Ocupacional": 91,
    "Posgrado de Segunda Especialidad en Enfermeria en Cardiologia": 64,
    "Posgrado de Segunda Especialidad de Enfermeria en Geriatria": 68,
    "Posgrado de Segunda Especialidad en Enfermeria en Gestion de Central de Esterilizacion": 80,
    "Segunda Especialidad de Enfermeria en Cardiologia": 64,
    "Segunda Especialidad de Enfermeria en Cirugia Cardiovascular": 65,
    "Segunda Especialidad de Enfermeria en Geriatria": 68,
    "Segunda Especialidad de Enfermeria en Nefrologia": 69,
    "Segunda Especialidad de Enfermeria en Pediatria": 84,
    "Segunda Especialidad de Enfermeria en Perfusion Extracorporea y Asistencia Circulatoria": 85,
    "Segunda Especialidad de Enfermeria en Cuidado de la Persona con Diabetes con Mencion en Educacion en Diabetes": 269,
    "Segunda Especialidad en Antiguo Testamento": 71,
    "Segunda Especialidad en Asesoria Genetica en Enfermeria Oncologica": 72,
    "Segunda Especialidad en Enfermeria en Gineco Obstetricia": 81,
    "Segunda Especialidad en Psicologia Clinica y de la Salud": 222,
    "Segunda Especialidad en Estudios Adventistas": 88,
    "Segunda Especialidad en Nuevo Testamento": 89,
    "Segunda Especialidad en Profecias Biblicas": 90,
    "Segunda Especialidad Profesional en Neuropsicologia": 272,
    "Segunda Especialidad Profesional en Psicooncologia y Cuidados Paliativos": 258,

    # MAESTRÍAS
    "Maestria Bim Facility Management": 273,
    "Maestria en Administracion de Negocios con mencion en Gestion Empresarial": 246,
    "Posgrado de Maestria en Auditoria Mencion: Auditoria Integral": 40,
    "Maestria en Auditoria con mencion en Auditoria Integral": 40,
    "Posgrado de Maestria en Ciencias de la Familia Mencion: Terapia Familiar": 238,
    "Maestria en Ciencias de la Familia": 238,
    "Maestria en Educacion con mencion en Curriculo": 230,
    "Maestria en Ingenieria Ambiental": 48,
    "Maestria en Ingenieria Ambiental y Desarrollo Sostenible": 247,
    "Maestria en Ingenieria de Sistemas con mencion en Ingenieria de Software": 231,
    "Maestria en Liderazgo Eclesiastico": 50,
    "Maestria en Misionologia": 51,
    "Maestria en Nutricion Humana con Mencion en Alimentacion Basada en Plantas": 270,
    "Maestria en Nutricion Humana con mencion en Nutricion Clinica": 52,
    "Maestria en Nutricion Humana con mencion en Nutricion Vegetariana": 232,
    "Maestria en Teologia": 56,
    "Posgrado de Maestria en Teologia": 56,
    "Maestria en Teologia Biblica": 233,
    "Maestria en Teologia Historica": 234,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Finanzas": 226,
    "Posgrado de Maestria en Administracion de Negocios con mencion en Liderazgo y Gestion Organizacional": 227,
    "Posgrado de Maestria en Administracion de Negocios con mencion en Recursos Humanos": 39,
    "Posgrado de Maestria en Educacion: Administracion Educativa": 42,
    "Posgrado de Maestria en Educacion: Investigacion y Docencia Universitaria": 228,
    "Posgrado de Maestria en Educacion: Mencion en Psicologia Educativa": 229,
    "Posgrado de Maestria en Enfermeria con Mencion en Administracion y Gestion": 46,
    "Posgrado de Maestria en Ingenieria de Sistemas con mencion en Direccion y Gestion de Tecnologias de Informacion": 49,
    "Posgrado de Maestria en Psicologia Clinica y de la Salud": 53,
    "Posgrado de Maestria en Salud Publica Mencion: Gestion de los Servicios de Salud": 223,
    "Posgrado de Maestria en Salud Publica Mencion: Salud Colectiva y Promocion de la Salud": 224,
    "Posgrado de Maestria en Terapia Familiar y de Pareja": 41,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Administracion de Recursos Humanos": 39,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Gestion Empresarial ": 246,
    "Posgrado de Maestria en Ciencias de la Familia Mencion: Terapia Familiar": 41,
    "Posgrado de Maestria en Ingenieria Ambiental": 48,
    "Posgrado de Maestria en Ingenieria de Sistemas : Direccion y Gestion de TI": 49,
    "Posgrado de Maestria en Nutricion Humana con Mencion en Nutricion Vegetariana": 232,
    "Posgrado de Maestria en Teologia Mencion: Liderazgo y Administracion Eclesiastica": 56,
    "Posgrado de Maestria en Misionologia": 51,
    "Posgrado de Maestria en Auditoria Mencion: Auditoria Integral": 40,
    "Posgrado de Maestria en Nutricion Humana con Mencion en Nutricion Vegetariana": 232,
    "Posgrado de Maestria en Teologia Biblica": 233,
    "Posgrado de Maestria en Administracion de Negocios Mencion: Gestion Empresarial": 246,

    # DOCTORADOS
    "Doctorado en Administracion de Negocios": 11,
    "Posgrado de Doctorado en Administracion de Negocios": 11,
    "Doctorado en Ciencias Contables y Financieras": 276,
    "Posgrado de Doctorado en Educacion mencion: Gestion Educativa": 235,
    "Doctorado en Educacion con mencion en Curriculo y Docencia": 12,
    "Posgrado de Doctorado en Enfermeria": 13,
    "Doctorado en Ingenieria de Sistemas": 14,
    "Posgrado de Doctorado en Salud Publica": 15,
    "Doctorado en Teologia": 16,
    "Posgrado de Doctorado en Teologia": 16,
    "Posgrado de Doctorado en Administracion de Negocios": 11,
    "Posgrado de Doctorado en Educacion mencion: Curriculo y Docencia": 12,
    "Posgrado de Doctorado en Ingenieria de Sistemas": 14,
    
}

# Realizar el reemplazo en la columna 'CODIGO_PROGRAMA_INGRESO'
df['CODIGO_PROGRAMA'] = df['CODIGO_PROGRAMA'].replace(reemplazos)

# Diccionario de modalidad de estudio
reemplazos_modalidad = {
    "Dato no declarado": 0,
    "Presencial": 1,
    "Semipresencial": 2,
    "A Distancia": 3
}

# Reemplazar MODALIDAD_ESTUDIO según el diccionario
df['MODALIDAD_ESTUDIO'] = df['MODALIDAD_ESTUDIO'].replace(reemplazos_modalidad)


#  Guardar archivo final
df.to_excel(nombre_salida, index=False)

print(f"\n✅ Archivo guardado como '{nombre_salida}'.")


✅ Archivo guardado como '/Users/jhonvilcaranatintaya/egresados_empr_final.xlsx'.


In [4]:
# =============================
# CORRECCIÓN DE CORREOS PERSONALES E INSTITUCIONALES
# =============================

import re

# Asegurar columnas necesarias
for col in ["CORREO_PERSONAL", "CORREO_INSTITUCIONAL", "NRO_DOCUMENTO"]:
    if col not in df.columns:
        df[col] = ""

# Normalizar columnas
df["NRO_DOCUMENTO"] = df["NRO_DOCUMENTO"].astype(str).str.strip()
df["CORREO_PERSONAL"] = df["CORREO_PERSONAL"].astype(str).str.strip().str.lower().replace({"nan": "", "none": ""})
df["CORREO_INSTITUCIONAL"] = df["CORREO_INSTITUCIONAL"].astype(str).str.strip().str.lower().replace({"nan": "", "none": ""})

# ==== 1. Correo Institucional ====

# Patrón válido: termina en @upeu.edu.pe
pat_institucional = r"^[a-z0-9._%+-]+@upeu\.edu\.pe$"

# Correos a corregir (vacíos o incorrectos)
mask_corr_inst = ~df["CORREO_INSTITUCIONAL"].str.match(pat_institucional, na=False)

# Asignar nuevo correo institucional basado en NRO_DOCUMENTO
df.loc[mask_corr_inst, "CORREO_INSTITUCIONAL"] = (
    "u" + df.loc[mask_corr_inst, "NRO_DOCUMENTO"] + "@upeu.edu.pe"
)

print(f"📧 Correos institucionales corregidos o generados: {mask_corr_inst.sum()}")


# ==== 2. Correo Personal ====

# Patrón válido: termina en @gmail.com
pat_gmail = r"^[a-z0-9._%+-]+@gmail\.com$"

# Correos a corregir (vacíos o incorrectos)
mask_corr_pers = ~df["CORREO_PERSONAL"].str.match(pat_gmail, na=False)

# Asignar nuevo correo personal basado en NRO_DOCUMENTO
df.loc[mask_corr_pers, "CORREO_PERSONAL"] = (
    "u" + df.loc[mask_corr_pers, "NRO_DOCUMENTO"] + "@gmail.com"
)
#  Guardar archivo final
df.to_excel(nombre_salida, index=False)

print(f"\n✅ Archivo guardado como '{nombre_salida}'.")
print(f"📧 Correos personales corregidos o generados: {mask_corr_pers.sum()}")

📧 Correos institucionales corregidos o generados: 21

✅ Archivo guardado como '/Users/jhonvilcaranatintaya/egresados_empr_final.xlsx'.
📧 Correos personales corregidos o generados: 34


In [6]:
import pandas as pd
import re

# Asegurar que existe la columna
if "CELULAR" not in df.columns:
    df["CELULAR"] = ""

# Función para limpiar y validar números de celular
def normalizar_celular(valor):
    if pd.isna(valor):
        return ""
    
    # Eliminar todo excepto números
    numero = re.sub(r'\D', '', str(valor))
    
    # Validar longitud (9 a 12 dígitos)
    if 9 <= len(numero) <= 12:
        return numero
    else:
        return ""  # Vacío si no cumple

# Aplicar limpieza
df["CELULAR"] = df["CELULAR"].apply(normalizar_celular)

# Reporte
cantidad_validos = df["CELULAR"].apply(lambda x: len(str(x)) in range(9, 13)).sum()
print(f"📱 Celulares válidos: {cantidad_validos} / {len(df)}")

📱 Celulares válidos: 33 / 34
